# Crop Health Monitoring — Nubaria Parcels (New Delta, Egypt)

**63 parcels** from `Nubarya_63_Parcels_GroundTruth.kml`.

| | |
|---|---|
| Block centre | 29.6712 E, 30.7796 N — Nubaria, Beheira |
| Block extent | 2,231 m × 1,342 m |
| Mapped area | **170.9 ha = 407.0 feddan** |
| Registered area | 173.5 ha = 413.1 feddan |

---

## Classifier corrected — six bugs fixed

The previous version returned **0 % disease** and almost no bare soil, and
painted the whole block yellow. Six faults, each verified numerically (a seventh,
found later, is described under the table):

| # | Bug | Effect |
|---|---|---|
| 1 | `(1.8 − SIPI)` **inverted** | A healthy pixel (SIPI 1.15) scored **0.65** for nutrient deficiency; a genuinely stressed one (SIPI 1.60) scored **0.20**. This single term drove the block yellow. |
| 2 | `BSI > 100` **unreachable** | BSI is a normalised difference, range −1…+1. Real values: dense crop −0.29, bare sand +0.20. Matched **zero** pixels, so bare soil could only come from `NDVI < 0.2`. |
| 3 | `PRI_Proxy` terms **always zero** | `(G−B)/(G+B)` is always positive over vegetation, so one nutrient and one disease component contributed nothing. |
| 4 | Disease **capped at 0.375** | Two bell curves peak at 0.25 each; with the dead PRI term disease could never win the argmax. |
| 5 | Healthy **unreachable** | Needed NDVI>0.7 **and** EVI>0.5 **and** SAVI>0.6 **and** NDRE>0.4 **and** GCI>5.0. A good crop at NDVI 0.72 scored 0.043 and lost to nutrient at 0.163. |
| 6 | Open water **cancelled** | `water_score[waterbody]=1.0` ran *before* `water_score[gen_bare]=0.0`, and every water pixel has NDVI<0.2 — so canals were classified bare soil. |
| 7 | Open water **mislabelled** | Fixing #6 put canals in the water-*stress* column, so they were reported as drought. They now get their own `open_water` class. |

Plus two in `compute_indices`:

* **NDRE used the wrong bands** — `(B6−B5)` instead of `(B8−B5)`. A healthy
  crop read 0.41 instead of 0.57, fell below the 0.35 threshold, and was
  flagged nitrogen-deficient.
* **PSRI was missing** — senescence is the only thing that separates disease
  from hunger.

### Verified on six realistic pixel profiles

| Pixel | NDVI | NDRE | NDMI | SIPI | PSRI | Result |
|---|---|---|---|---|---|---|
| healthy maize | 0.78 | 0.57 | 0.36 | 1.04 | 0.06 | **healthy** ✓ |
| nitrogen deficient | 0.57 | 0.21 | 0.18 | 1.19 | 0.19 | **nutrient def** ✓ |
| water stressed | 0.58 | 0.36 | −0.09 | 1.14 | 0.13 | **water stress** ✓ |
| diseased / dying | 0.37 | 0.22 | 0.00 | 1.47 | 0.32 | **disease** ✓ |
| bare calcareous | 0.12 | 0.06 | −0.10 | 2.00 | 0.41 | **bare soil** ✓ |
| canal water | −0.43 | −0.33 | 0.33 | 1.33 | −0.33 | **open water** ✓ (own class, not water stress) |

**6 / 6 correct.** All seven classes now reachable. (These six pixels are hand-built
illustrations, not a validation: the class scores are uncalibrated rule weights.)

---

### Calibration for this site

```python
SOIL_CLASS = 'calcareous_sandy'   # Nubaria reclaimed desert
# season is chosen automatically from the Sentinel-2 date (Nov-Mar winter, else summer)
```

Reclaimed calcareous desert is far brighter in SWIR than old-Delta clay, and
runs several degrees hotter. Using Delta thresholds here labels healthy crop as
bare soil. The soil constant sits at the top of cell 8; the season follows the image date.

In [ ]:
# Run once per Colab session
!pip -q install geopandas rasterio geemap earthengine-api folium contextily adjustText arabic-reshaper python-bidi --upgrade


In [ ]:
import os, io, glob, json, base64, warnings
warnings.filterwarnings('ignore')

import ee
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import rasterio
from rasterio.warp import reproject, Resampling
from pyproj import Transformer
from shapely.geometry import box, mapping
import folium
from folium.raster_layers import ImageOverlay
from folium import IFrame
import matplotlib as mpl


In [ ]:
# Earth Engine project: set EE_PROJECT_ID in the environment, or edit the default.
EE_PROJECT_ID = os.environ.get('EE_PROJECT_ID', 'larvest-zeyad')
try:
    ee.Initialize(project=EE_PROJECT_ID)          # works silently when credentials are cached
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT_ID)


In [ ]:
# ════════════════════════════════════════════════════════════════
#  LOAD PARCELS FROM THE GROUND-TRUTH KML
#
#  Replaces the embedded coordinate CSV. The KML carries everything the
#  CSV had, plus owner names, registered (Excel) areas, owner counts and
#  a geometry-vs-registered QA field.
#
#  Upload Nubarya_63_Parcels_GroundTruth.kml when prompted.
# ════════════════════════════════════════════════════════════════
import re, html
import xml.etree.ElementTree as ET
from shapely.geometry import Polygon, MultiPolygon
from shapely.ops import transform as shp_transform
import pyproj

KML_PATH = 'Nubarya_63_Parcels_GroundTruth.kml'
KML_NS   = {'k': 'http://www.opengis.net/kml/2.2'}

if not os.path.exists(KML_PATH):
    from google.colab import files
    uploaded = files.upload()
    cand = [k for k in uploaded if k.lower().endswith('.kml')]
    if not cand:
        raise FileNotFoundError('No .kml file was uploaded.')
    KML_PATH = cand[0]


def _kml_coords(text):
    pts = []
    for tok in text.replace('\n', ' ').split():
        p = tok.split(',')
        if len(p) >= 2:
            pts.append((float(p[0]), float(p[1])))
    return pts


def _owner_from_description(desc):
    """
    Owner names are NOT in <ExtendedData> — they exist only inside the HTML
    popup. If the KML is ever regenerated with a different popup layout,
    this regex is the thing to re-check.
    """
    if not desc:
        return None
    m = re.search(r'Owner\(s\)</td>\s*<td[^>]*>(.*?)</td>', desc, re.S | re.I)
    if not m:
        return None
    txt = re.sub(r'<[^>]+>', '', m.group(1))
    return html.unescape(txt).strip() or None


def load_parcels_from_kml(path, verbose=True):
    root = ET.parse(path).getroot()
    recs = []

    for pm in root.iter('{http://www.opengis.net/kml/2.2}Placemark'):
        rec = {}
        nm = pm.find('k:name', KML_NS)
        rec['kml_name'] = nm.text.strip() if nm is not None and nm.text else None

        de = pm.find('k:description', KML_NS)
        rec['owner'] = _owner_from_description(de.text if de is not None else None)

        for d in pm.iter('{http://www.opengis.net/kml/2.2}Data'):
            key = d.get('name')
            v = d.find('k:value', KML_NS)
            if key and v is not None and v.text is not None:
                rec[key] = v.text.strip()

        polys = []
        for poly in pm.iter('{http://www.opengis.net/kml/2.2}Polygon'):
            ob = poly.find('.//k:outerBoundaryIs//k:coordinates', KML_NS)
            if ob is None or not ob.text:
                continue
            shell = _kml_coords(ob.text)
            holes = [_kml_coords(c.text) for c in
                     poly.findall('.//k:innerBoundaryIs//k:coordinates', KML_NS)
                     if c is not None and c.text]
            if len(shell) >= 4:
                polys.append(Polygon(shell, holes))
        if not polys:
            continue

        geom = polys[0] if len(polys) == 1 else MultiPolygon(polys)
        if not geom.is_valid:
            geom = geom.buffer(0)
        rec['geometry'] = geom
        recs.append(rec)

    gdf = gpd.GeoDataFrame(recs, crs='EPSG:4326')

    for c in ['parcel_id', 'owner_count', 'excel_area_m2', 'excel_area_km2',
              'geometry_area_m2', 'geometry_area_km2', 'geometry_vs_excel_pct']:
        if c in gdf.columns:
            gdf[c] = pd.to_numeric(gdf[c], errors='coerce')

    if 'parcel_id' not in gdf.columns or gdf['parcel_id'].isna().any():
        gdf['parcel_id'] = gdf['kml_name'].str.extract(r'(\d+)')[0].astype(float)
    gdf['parcel_id'] = gdf['parcel_id'].astype(int)

    _eq = pyproj.Transformer.from_crs('EPSG:4326', 'EPSG:6933',
                                      always_xy=True).transform
    gdf['area_ha_polygon']     = [shp_transform(_eq, g).area / 1e4 for g in gdf.geometry]
    gdf['area_feddan_polygon'] = gdf['area_ha_polygon'] / 0.42

    if 'excel_area_m2' in gdf.columns:
        gdf['area_ha_excel']     = gdf['excel_area_m2'] / 1e4
        gdf['area_feddan_excel'] = gdf['area_ha_excel'] / 0.42
    else:
        gdf['area_ha_excel'] = gdf['area_feddan_excel'] = np.nan

    # Satellite statistics come from PIXELS INSIDE THE POLYGON, so every
    # per-hectare figure must use the polygon area. The registered Excel
    # area is carried alongside for reporting to the owner.
    gdf['area_ha']     = gdf['area_ha_polygon']
    gdf['area_feddan'] = gdf['area_feddan_polygon']

    cen = gdf.geometry.to_crs('EPSG:6933').centroid.to_crs('EPSG:4326')
    gdf['lat'], gdf['lon'] = cen.y, cen.x

    # 'ada' is the number printed on the map
    gdf['parcel_uid'] = gdf['parcel_id'].map(lambda i: f'P{i:03d}')
    gdf['ada']        = gdf['parcel_id'].astype(str)
    gdf['parsel']     = '1'
    gdf['mahalle']    = 'Nubaria'
    gdf['mevkii']     = 'New Delta'
    gdf['nitelik']    = 'agricultural land'

    gdf = gdf.sort_values('parcel_id').reset_index(drop=True)

    if verbose:
        print(f'Parcels from KML   : {len(gdf)}')
        print(f'Owners named       : {gdf.owner.notna().sum()} / {len(gdf)}'
              f'   ({gdf.owner.nunique()} unique)')
        print(f'Polygon area total : {gdf.area_ha_polygon.sum():.1f} ha'
              f' = {gdf.area_feddan_polygon.sum():.1f} feddan')
        if gdf.area_ha_excel.notna().any():
            print(f'Registered total   : {gdf.area_ha_excel.sum():.1f} ha'
                  f' = {gdf.area_feddan_excel.sum():.1f} feddan')
        print(f'Invalid geometries : {int((~gdf.geometry.is_valid).sum())}')
        print(f'Duplicate IDs      : {int(gdf.parcel_id.duplicated().sum())}')
        minx, miny, maxx, maxy = gdf.total_bounds
        bw = (maxx - minx) * 111320 * np.cos(np.radians(gdf.lat.mean()))
        bh = (maxy - miny) * 110574
        print(f'Block extent       : {bw:.0f} m x {bh:.0f} m')
    return gdf


parcels_gdf = load_parcels_from_kml(KML_PATH)

# ---- data-quality review -------------------------------------------
if 'geometry_vs_excel_pct' in parcels_gdf.columns:
    d = parcels_gdf['geometry_vs_excel_pct']
    print(f'\nGeometry vs registered : median {d.median():+.2f}%  '
          f'range {d.min():+.2f}% to {d.max():+.2f}%')
    flagged = parcels_gdf.loc[d.abs() > 10]
    if len(flagged):
        print(f'\n  {len(flagged)} parcel(s) differ by more than 10% — the drawn'
              f' boundary may not match the registered plot:')
        for _, r in flagged.iterrows():
            print(f'    parcel {r.ada:>3s}  registered {r.excel_area_m2:>9,.0f} m2'
                  f'   mapped {r.geometry_area_m2:>9,.0f} m2'
                  f'   {r.geometry_vs_excel_pct:+7.2f}%')

if 'owner_count' in parcels_gdf.columns:
    multi = parcels_gdf[parcels_gdf['owner_count'] > 1]
    if len(multi):
        print(f'\n{len(multi)} parcel(s) have more than one registered owner — a'
              f' satellite result covers the whole polygon, not each share:')
        print('    parcels ' + ', '.join(multi['ada']))

if not (24 <= parcels_gdf.lat.mean() <= 32 and 24 <= parcels_gdf.lon.mean() <= 37):
    print('\n  WARNING: block centroid is outside Egypt — check the KML.')

_show = [c for c in ['ada', 'owner', 'owner_count', 'area_feddan_excel',
                     'area_feddan_polygon', 'geometry_vs_excel_pct'] if c in parcels_gdf.columns]
parcels_gdf[_show].head(10)

In [ ]:
# ---- Configuration ----
# Egyptian summer season (maize / rice / cotton run roughly May to September).
# For a winter crop (wheat, sugar beet, berseem) use Nov 01 -> Feb 28.
START_DATE = '2025-05-15'
END_DATE   = '2025-09-15'

BLOCK_BUFFER_M = 120     # buffer around the WHOLE block before downloading
SCALE_M        = 10

S2_BANDS   = ['B1','B2','B3','B4','B5','B6','B7','B8','B8A','B9','B11','B12']
CLOUD_THRESHOLDS_S2  = [10, 20, 40, 60, 90]     # relaxed progressively
CLOUD_THRESHOLDS_LS8 = [10, 20, 40, 60, 90]    # (unused: thermal now picks the scene nearest in time)
LST_MAX_GAP_DAYS     = 8      # accept a Landsat scene this close to the Sentinel-2 date

# 63 parcels x 19 indices would write ~1,200 PNG files.
# Leave False unless you specifically want per-index cartographic maps.
MAKE_INDEX_MAPS = False

OUT_DIR   = './nubaria_parcels_data'
BLOCK_DIR = os.path.join(OUT_DIR, '_block')      # shared imagery lives here
os.makedirs(BLOCK_DIR, exist_ok=True)

def meters_to_deg(m, lat):
    return m / (111320 * np.cos(np.radians(lat)))

print(f'Search window   : {START_DATE} to {END_DATE}')
print(f'Block buffer    : {BLOCK_BUFFER_M} m')
print(f'Per-index maps  : {MAKE_INDEX_MAPS}')
print(f'Output folder   : {OUT_DIR}')

In [ ]:
def get_least_cloudy_image(collection_id, aoi, start, end, cloud_prop, thresholds):
    """Return (ee.Image, cloud_pct_used) for the least-cloudy scene, relaxing the
    cloud filter progressively until something is found, or (None, None)."""
    for thr in thresholds:
        coll = (ee.ImageCollection(collection_id)
                 .filterBounds(aoi)
                 .filterDate(start, end)
                 .filter(ee.Filter.lt(cloud_prop, thr))
                 .sort(cloud_prop))
        if coll.size().getInfo() > 0:
            img = ee.Image(coll.first())
            return img, thr
    return None, None


def get_sentinel1_image(aoi, start, end):
    for orbit in ['DESCENDING', 'ASCENDING']:
        coll = (ee.ImageCollection('COPERNICUS/S1_GRD')
                 .filterBounds(aoi)
                 .filterDate(start, end)
                 .filter(ee.Filter.eq('instrumentMode', 'IW'))
                 .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
                 .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
                 .filter(ee.Filter.eq('orbitProperties_pass', orbit))
                 .sort('system:time_start', False))
        if coll.size().getInfo() > 0:
            img = ee.Image(coll.first())
            for band in ['VV', 'VH']:
                filtered = img.select(band).focal_mean(radius=1, kernelType='square', units='pixels')
                img = img.addBands(filtered.rename(band), overwrite=True)
            return img
    return None


def download_bands(image, bands, aoi, scale, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    region = aoi.bounds().getInfo()['coordinates']
    paths = {}
    for b in bands:
        try:
            url = image.select(b).getDownloadURL({'scale': scale, 'region': region, 'format': 'GEO_TIFF'})
            r = requests.get(url, timeout=120)
            if r.status_code == 200:
                p = os.path.join(out_dir, f'{b}.tif')
                with open(p, 'wb') as f:
                    f.write(r.content)
                paths[b] = p
            else:
                print(f'    failed to download band {b} (HTTP {r.status_code})')
        except Exception as e:
            print(f'    error downloading band {b}: {e}')
    return paths


def get_landsat_lst_image(aoi, target_date, max_gap_days=8, max_cloud=70):
    """
    Land-surface temperature (deg C) from Landsat 8/9 Collection-2 LEVEL-2 (ST_B10),
    taken from the scene closest in time to `target_date`.

    Returns (ee.Image band 'LST_C' with -999 as no-data, sensing date, gap in days)
    or (None, None, None).  Cloud / shadow pixels and pixels with ST uncertainty
    above 3 K are masked.  Unlike the old top-of-atmosphere brightness temperature
    this is a real surface temperature, so the 34 / 42 C canopy thresholds mean
    what they say.
    """
    t0 = ee.Date(target_date)
    cols = [ee.ImageCollection(c).filterBounds(aoi)
              .filterDate(t0.advance(-max_gap_days, 'day'), t0.advance(max_gap_days + 1, 'day'))
              .filter(ee.Filter.lt('CLOUD_COVER', max_cloud))
            for c in ('LANDSAT/LC08/C02/T1_L2', 'LANDSAT/LC09/C02/T1_L2')]
    coll = cols[0].merge(cols[1])
    coll = coll.map(lambda i: i.set('dt', ee.Number(i.get('system:time_start'))
                                           .subtract(t0.millis()).abs())).sort('dt')
    if coll.size().getInfo() == 0:
        return None, None, None
    img = ee.Image(coll.first())
    sensed = pd.Timestamp(img.date().format('YYYY-MM-dd').getInfo())
    gap = abs((sensed - pd.Timestamp(target_date)).days)
    cloudy = img.select('QA_PIXEL').bitwiseAnd(0b11110).neq(0)      # dilated, cirrus, cloud, shadow
    lst = (img.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15)
              .updateMask(cloudy.Not())
              .updateMask(img.select('ST_QA').multiply(0.01).lte(3.0))
              .resample('bilinear').rename('LST_C').unmask(-999))
    return lst, sensed.strftime('%Y-%m-%d'), int(gap)


In [ ]:
def resample_to_reference(src_path, ref_path, out_path, resampling=Resampling.bilinear, src_nodata=None):
    with rasterio.open(ref_path) as ref:
        ref_transform, ref_crs = ref.transform, ref.crs
        ref_w, ref_h = ref.width, ref.height

    with rasterio.open(src_path) as src:
        dst = np.full((ref_h, ref_w), np.nan, dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=ref_transform, dst_crs=ref_crs,
            src_nodata=src_nodata, dst_nodata=np.nan,     # no-data is skipped, never blended into valid pixels
            resampling=resampling)

    with rasterio.open(ref_path) as ref:
        meta = ref.meta.copy()
    meta.update(dtype='float32', count=1, nodata=np.nan)
    with rasterio.open(out_path, 'w', **meta) as dst_ds:
        dst_ds.write(dst, 1)
    return out_path


def bands_to_dataframe(paths, ref_key='B4'):
    with rasterio.open(paths[ref_key]) as ref:
        h, w = ref.height, ref.width
        transform, crs = ref.transform, ref.crs

    data = {}
    for name, p in paths.items():
        with rasterio.open(p) as src:
            arr = src.read(1).astype(np.float32)
            if arr.shape != (h, w):
                print(f'    shape mismatch for {name}, skipping')
                continue
            if src.nodata is not None:
                arr[arr == src.nodata] = np.nan
            data[name] = arr.flatten()

    df = pd.DataFrame(data)
    rows, cols = np.unravel_index(np.arange(h * w), (h, w))
    xs = transform.c + (cols + 0.5) * transform.a + (rows + 0.5) * transform.b
    ys = transform.f + (cols + 0.5) * transform.d + (rows + 0.5) * transform.e
    if crs.is_geographic:
        lon, lat = xs, ys
    else:
        tr = Transformer.from_crs(crs, 'EPSG:4326', always_xy=True)
        lon, lat = tr.transform(xs, ys)
    df['longitude'] = lon
    df['latitude'] = lat
    return df, (h, w)


In [ ]:
def dn_to_temperature(dn_values, ML=0.0003342, AL=0.1):
    """
    LEGACY - no longer used by the pipeline (cell 11 now downloads Landsat Level-2
    land-surface temperature directly). Kept only for reference.

    Landsat 8 Band 10 DN -> brightness temperature (Celsius).

    LC08/C02/T1 is top-of-atmosphere, so this is BRIGHTNESS temperature, not
    land surface temperature. Relative patterns across the block are reliable;
    absolute values are not. For true LST use LC08/C02/T1_L2 and ST_B10.
    """
    dn = np.asarray(dn_values, dtype=float)
    radiance = ML * dn + AL
    with np.errstate(divide='ignore', invalid='ignore'):
        temp_k = 1321.08 / np.log((774.89 / radiance) + 1.0)
    return temp_k - 273.15


def compute_indices(df):
    """
    FIXES vs the previous version
    ------------------------------------------------------------------
    NDRE   was (B6-B5)/(B6+B5).  NDRE is NIR vs RedEdge1, so it must be
           (B8-B5)/(B8+B5).  The old formula compressed the range: a healthy
           crop read ~0.41 instead of ~0.57, fell below the 0.35 threshold,
           and pushed the whole block into 'nutrient deficiency'.

    PSRI   added.  Senescence is the ONLY thing that separates disease from
           hunger, and it was missing entirely.

    ARI    was byte-identical to CRI1_Proxy (both 1/G - 1/RE1).  CRI1 is now
           the real formula (1/B - 1/G).

    SIPI   the divide-by-zero guard ran BEFORE the column existed and was
           immediately overwritten, so it did nothing.

    Scaling is now idempotent: re-running this cell no longer divides the
    bands a second time.
    """
    optical_bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B8', 'B11', 'B12']
    for b in optical_bands:
        if b in df.columns:
            raw = df[b].astype(float)
            # if the column is already reflectance, do not scale it again
            ref = raw * 0.0001 if raw.max() > 1.5 else raw
            invalid = (ref <= 0) | (ref > 1) | ref.isna()
            ref = ref.where(~invalid, np.nan)
            df[f'{b}_ref'] = ref

    with np.errstate(divide='ignore', invalid='ignore'):
        B, G, R = df['B2_ref'], df['B3_ref'], df['B4_ref']
        RE1, RE2 = df['B5_ref'], df['B6_ref']
        N, SWIR = df['B8_ref'], df['B11_ref']
        SWIR2 = df['B12_ref'] if 'B12_ref' in df.columns else SWIR

        # ---- vigour -------------------------------------------------
        df['NDVI']  = (N - R) / (N + R)
        df['EVI']   = 2.5 * (N - R) / (N + 6.0 * R - 7.5 * B + 1.0)
        df['SAVI']  = ((N - R) / (N + R + 0.5)) * 1.5
        df['OSAVI'] = (N - R) / (N + R + 0.16)
        term = np.sqrt(np.maximum((2 * N + 1) ** 2 - 8 * (N - R), 0))
        df['MSAVI'] = 0.5 * ((2 * N + 1) - term)
        df['GNDVI'] = (N - G) / (N + G)

        # ---- water --------------------------------------------------
        df['NDWI'] = (G - N) / (G + N)          # OPEN WATER mask only
        df['NDMI'] = (N - SWIR) / (N + SWIR)    # true crop water stress
        df['MSI']  = (SWIR / N).clip(0, 3)
        df['GVMI'] = (N + 0.1 - SWIR2 + 0.02) / (N + 0.1 + SWIR2 + 0.02)

        # ---- nitrogen / chlorophyll ---------------------------------
        df['NDRE']  = (N - RE1) / (N + RE1)          # FIXED
        df['NDRE2'] = (RE2 - RE1) / (RE2 + RE1)      # old formula, not scored
        df['CIre']  = (N / RE1) - 1
        df['GCI']   = (N / G) - 1
        df['MCARI'] = ((RE1 - R) - 0.2 * (RE1 - G)) * (RE1 / R)

        # ---- pigment / senescence -----------------------------------
        den = N - R
        df['SIPI'] = ((N - B) / den.where(den.abs() > 1e-6)).clip(0, 2)
        df['PSRI'] = (R - B) / RE2                   # NEW
        df['ARI']  = ((1.0 / G) - (1.0 / RE1)).clip(-5, 20)
        df['CRI1_Proxy'] = ((1.0 / B) - (1.0 / G)).clip(0, 30)   # real CRI1

        # ---- soil ----------------------------------------------------
        df['BI']  = np.sqrt((R ** 2 + N ** 2) / 2)
        df['BSI'] = ((SWIR + R) - (N + B)) / ((SWIR + R) + (N + B))

        # Kept for continuity but NEVER scored: (G-B)/(G+B) is always positive
        # over vegetation, so every threshold built on it was dead code.
        df['PRI_Proxy'] = (G - B) / (G + B)

    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    # Idempotent: only scale bands still on the DN scale.
    for band in ['B1','B2','B3','B4','B5','B6','B7','B8','B8A','B9','B11','B12']:
        if band in df.columns:
            mx = df[band].max()
            if pd.notna(mx) and mx > 1.5:
                df[band] = df[band].astype(float) / 10000.0
    return df


# Soil brightness thresholds. Reclaimed calcareous desert is far brighter in
# SWIR than old-Delta clay; clay values here would label healthy crop as bare.
SOIL_THRESHOLDS = {
    'old_delta_clay':   {'B11': 0.22, 'B12': 0.18, 'BSI': 0.05},
    'reclaimed_sandy':  {'B11': 0.30, 'B12': 0.26, 'BSI': 0.12},
    'calcareous_sandy': {'B11': 0.34, 'B12': 0.30, 'BSI': 0.15},
}
SOIL_CLASS = 'calcareous_sandy'          # Nubaria / New Delta

# Canopy temperature bands, Celsius. Reclaimed desert runs hotter than Delta.
THERMAL_BANDS = {
    'winter': {'mild': 30.0, 'severe': 38.0},
    'summer': {'mild': 34.0, 'severe': 42.0},
}
SEASON = 'summer'                         # default only; cell 11 passes the season of the image date


def season_for_date(date_str):
    """Egypt: November-March is the cool season, April-October the hot one."""
    m = int(str(date_str)[5:7])
    return 'winter' if (m >= 11 or m <= 3) else 'summer'


def classify_crop_issues(df, handle_null=True, soil_class=SOIL_CLASS,
                         season=SEASON, verbose=False):
    """
    Corrected classifier.  Every fix below was verified numerically.

    1  SIPI WAS INVERTED
       (1.8 - SIPI) gave a healthy pixel (SIPI 1.15) a nutrient score of 0.65
       and a genuinely stressed one (SIPI 1.60) only 0.20.  Backwards.  This
       single term is what painted the whole block yellow.

    2  BSI > 100 WAS UNREACHABLE
       BSI is a normalised difference, range -1..+1.  Real values: dense crop
       -0.29, bare sand +0.20.  prim_bare matched ZERO pixels and one soil
       component was permanently 0, so bare soil could only ever come from
       NDVI < 0.2.

    3  PRI_Proxy TERMS WERE DEAD
       (G-B)/(G+B) is always positive over vegetation, so (0.0 - PRI) and
       (-0.05 - PRI) were permanently 0 — one nutrient and one disease
       component contributed nothing at all.

    4  DISEASE COULD NEVER WIN
       Two bell-curve components peak at 0.25 each; with the dead PRI term the
       maximum possible disease score was (0.25+0.25+1.0+0)/4 = 0.375.

    5  HEALTHY WAS UNREACHABLE
       Required NDVI>0.7 AND EVI>0.5 AND SAVI>0.6 AND NDRE>0.4 AND GCI>5.0.
       A good crop at NDVI 0.72 scored 0.043 and lost to nutrient at 0.163.

    6  OPEN WATER WAS CANCELLED
       water_score[waterbody] = 1.0 ran BEFORE water_score[gen_bare] = 0.0, and
       every water pixel has NDVI < 0.2, so canals were classified bare soil.

    7  OPEN WATER WAS MISLABELLED
       Fix 6 put canals in the water-STRESS column, so they were reported as
       drought. They now get their own class, 'open_water'.
    """
    required = ['NDVI','EVI','SAVI','NDWI','NDRE','MSI','SIPI','PSRI','ARI',
                'thermal','B4','B8','B11','B12','GCI','NDMI','BSI','CIre','GVMI']
    for c in required:
        if c not in df.columns:
            df[c] = np.nan

    def g(c):
        return df[c] if c in df.columns else pd.Series(np.nan, index=df.index)

    SOIL   = SOIL_THRESHOLDS[soil_class]
    t_mild = THERMAL_BANDS[season]['mild']
    t_sev  = THERMAL_BANDS[season]['severe']

    def ramp(s, lo, hi):
        s = np.asarray(s, dtype=float)
        return np.zeros_like(s) if hi == lo else np.clip((s - lo) / (hi - lo), 0, 1)

    def inv(s, hi, lo):
        s = np.asarray(s, dtype=float)
        return np.zeros_like(s) if hi == lo else np.clip((hi - s) / (hi - lo), 0, 1)

    def calc_score(lst):
        arr = np.vstack([np.asarray(x, dtype=float) for x in lst])
        sums = np.nansum(arr, axis=0)
        cnts = (~np.isnan(arr)).sum(axis=0)
        out = np.zeros_like(sums)
        m = cnts > 0
        out[m] = sums[m] / cnts[m]
        return out

    # ---- HEALTHY  (FIX 5: thresholds at realistic mid-season values) ----
    healthy_score = calc_score([
        ramp(g('NDVI'), 0.45, 0.80),
        ramp(g('EVI'),  0.25, 0.60),
        ramp(g('SAVI'), 0.35, 0.65),
        ramp(g('NDRE'), 0.20, 0.45),
        ramp(g('GCI'),  2.0,  6.0),
        ramp(g('NDMI'), 0.10, 0.35),
    ])

    # ---- BARE SOIL  (FIX 2: real BSI thresholds) -----------------------
    soil_score = calc_score([
        inv(g('NDVI'), 0.25, 0.05),
        inv(g('SAVI'), 0.22, 0.05),
        ramp(g('BSI'), SOIL['BSI'] - 0.10, SOIL['BSI'] + 0.15),
        ramp(g('B11'), SOIL['B11'] - 0.05, SOIL['B11'] + 0.12),
        ramp(g('B12'), SOIL['B12'] - 0.05, SOIL['B12'] + 0.12),
    ])

    # ---- WATER STRESS --------------------------------------------------
    # NDMI counted twice: it is the one index that separates thirst from
    # hunger and must not be outvoted by weaker terms.
    water_score = calc_score([
        ramp(g('thermal'), t_mild, t_sev),
        inv(g('NDMI'), 0.20, -0.10),
        inv(g('NDMI'), 0.20, -0.10),
        ramp(g('MSI'), 0.9, 1.8),
        inv(g('GVMI'), 0.30, 0.00),
        ramp(g('B11'), SOIL['B11'] * 0.75, SOIL['B11'] * 1.15),
    ])

    # ---- NUTRIENT  (FIX 1 and 3) ---------------------------------------
    nutrient_score = calc_score([
        inv(g('NDRE'), 0.35, 0.12),
        inv(g('GCI'),  5.0,  1.5),
        inv(g('CIre'), 3.0,  0.5),
        ramp(g('SIPI'), 1.05, 1.70),        # HIGH SIPI = stress
    ])

    # A nitrogen-starved plant is PALE BUT ALIVE, so PSRI stays low. Rising
    # PSRI means tissue is actually dying, which is disease, not hunger.
    # Without this damping the two classes are nearly identical.
    nutrient_score = nutrient_score * (1.0 - 0.55 * ramp(g('PSRI'), 0.18, 0.38))

    # ---- DISEASE  (FIX 3 and 4) ----------------------------------------
    # NDRE is deliberately EXCLUDED: low NDRE is the nutrient signature, and
    # including it made the two classes indistinguishable.  PSRI is weighted
    # twice because senescence is what actually separates them.
    disease_score = calc_score([
        inv(g('NDVI'), 0.75, 0.30),
        ramp(g('PSRI'), 0.18, 0.38),
        ramp(g('PSRI'), 0.18, 0.38),
        ramp(g('ARI'),  2.0,  7.0),
        inv(g('EVI'), 0.55, 0.20),
    ])

    # ================= OVERRIDES, in order ==============================
    ndvi_v = g('NDVI').values
    ndwi_v = g('NDWI').values
    bsi_v  = g('BSI').values

    waterbody = ndwi_v > 0.30

    # Bare floor capped at 0.25: a diseased crop can fall to NDVI 0.35-0.45
    # and must not be swallowed by the soil class.
    BARE_FLOOR = 0.20
    gen_bare = (ndvi_v < BARE_FLOOR) & ~waterbody

    for arr in (nutrient_score, disease_score, healthy_score):
        arr[gen_bare] = 0.0
    water_score[gen_bare] = 0.0
    soil_score[gen_bare]  = 1.0

    prim_bare = (ndvi_v < 0.12) & (bsi_v > SOIL['BSI']) & ~waterbody
    soil_score[prim_bare] = 1.0

    # FIX 6: open water applied LAST so the bare override cannot cancel it.
    for arr in (soil_score, nutrient_score, disease_score, healthy_score):
        arr[waterbody] = 0.0
    water_score[waterbody] = 1.0

    if verbose:
        print(f'    open water            {int(waterbody.sum()):>9,}')
        print(f'    bare (NDVI<{BARE_FLOOR})      {int(gen_bare.sum()):>9,}')
        print(f'    bare + BSI confirmed  {int(prim_bare.sum()):>9,}')

    # ================= NORMALISE =========================================
    stack = np.nan_to_num(np.vstack([soil_score, water_score, nutrient_score,
                                     disease_score, healthy_score]).T)
    probs = stack / (stack.sum(axis=1, keepdims=True) + 1e-9)
    df[['soil_prob','water_prob','nutrient_prob','disease_prob','healthy_prob']] = probs

    df['empty_prob'] = 0.0
    df.loc[stack.sum(axis=1) < 0.08, 'empty_prob'] = 0.5

    if handle_null:
        core = [c for c in ['NDVI','EVI','SAVI','NDWI','NDRE','thermal',
                            'B4','B8','B11'] if c in df.columns]
        many_nan = df[core].isnull().sum(axis=1) >= (len(core) // 2 + 1)
        df.loc[many_nan, ['soil_prob','water_prob','nutrient_prob',
                          'disease_prob','healthy_prob']] = 0
        df.loc[many_nan, 'empty_prob'] = 1.0

    allc = ['soil_prob','water_prob','nutrient_prob',
            'disease_prob','healthy_prob','empty_prob']
    arr = np.nan_to_num(df[allc].values)
    final = arr / (arr.sum(axis=1, keepdims=True) + 1e-9)
    df[allc] = final

    labels = np.array(['bare_soil','water_stress','nutrient_def or low_veg',
                       'disease','healthy','empty'], dtype=object)
    idx = np.argmax(final, axis=1)
    pred = labels[idx]
    pred[(idx == 1) & waterbody] = 'open_water'      # FIX 7: canals are not drought
    df['prediction'] = pred
    df['pred_prob']  = final[np.arange(len(df)), idx]

    # Pixels the model is unsure about, plus high-severity calls worth
    # verifying on the ground before anything is reported.
    severe = df['prediction'].isin(['disease', 'water_stress'])
    df['needs_scouting'] = (df['pred_prob'] < 0.35) | (severe & (df['pred_prob'] > 0.55))
    return df


CLASS_COLORS = {
    'empty':                    '#808080',
    'bare_soil':                '#D2B48C',
    'healthy':                  '#00A000',
    'open_water':               '#00C2C7',
    'water_stress':             '#0050FF',
    'nutrient_def or low_veg':  '#FFD700',
    'disease':                  '#FF0000',
}



In [ ]:
from matplotlib.ticker import FuncFormatter

def _deg_to_dm(deg, is_lon):
    """Format a decimal-degree coordinate as DD°MM.mmm'H, e.g. 30°43.200'E."""
    hemi = ('E' if deg >= 0 else 'W') if is_lon else ('N' if deg >= 0 else 'S')
    d = abs(deg)
    dd = int(d)
    mm = (d - dd) * 60
    return f"{dd}\u00b0{mm:.3f}'{hemi}"


def _add_north_arrow(ax):
    ax.annotate('N', xy=(0.95, 0.95), xycoords='axes fraction', fontsize=13,
                ha='center', va='center', fontweight='bold', zorder=6)
    ax.annotate('', xy=(0.95, 0.93), xytext=(0.95, 0.80), xycoords='axes fraction',
                arrowprops=dict(arrowstyle='-|>', color='black', lw=2), zorder=6)


def _setup_cartographic_axes(overlay_rgba, lonmin, lonmax, latmin, latmax, title,
                              context_factor=1.5, figsize=(9, 7.5)):
    pad_lon = max((lonmax - lonmin) * context_factor, 0.0004)
    pad_lat = max((latmax - latmin) * context_factor, 0.0004)
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(lonmin - pad_lon, lonmax + pad_lon)
    ax.set_ylim(latmin - pad_lat, latmax + pad_lat)
    try:
        import contextily as cx
        cx.add_basemap(ax, crs='EPSG:4326', source=cx.providers.Esri.WorldImagery, attribution=False)
    except Exception as e:
        print(f'    basemap not added ({e})')
    ax.imshow(overlay_rgba, extent=[lonmin, lonmax, latmin, latmax], origin='upper',
              interpolation='none', zorder=3)
    ax.set_title(title, fontsize=13, pad=14)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, p: _deg_to_dm(v, True)))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: _deg_to_dm(v, False)))
    plt.setp(ax.get_xticklabels(), fontsize=8)
    plt.setp(ax.get_yticklabels(), fontsize=8)
    ax.grid(True, color='white', linewidth=0.4, alpha=0.6, zorder=4)
    _add_north_arrow(ax)
    return fig, ax


def cartographic_categorical_map(overlay_rgba, lonmin, lonmax, latmin, latmax, title,
                                  legend_handles, out_path, context_factor=1.5):
    fig, ax = _setup_cartographic_axes(overlay_rgba, lonmin, lonmax, latmin, latmax, title,
                                       context_factor=context_factor)
    ax.legend(handles=legend_handles, bbox_to_anchor=(1.02, 1), loc='upper left',
              frameon=True, title='Legend')
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.show()
    plt.close(fig)


def cartographic_continuous_map(overlay_rgba, lonmin, lonmax, latmin, latmax, title,
                                 cmap_name, vmin, vmax, cbar_label, out_path, context_factor=1.5):
    fig, ax = _setup_cartographic_axes(overlay_rgba, lonmin, lonmax, latmin, latmax, title,
                                       context_factor=context_factor)
    sm = mpl.cm.ScalarMappable(cmap=cmap_name, norm=mpl.colors.Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])
    fig.colorbar(sm, ax=ax, fraction=0.045, pad=0.03, label=cbar_label)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.show()
    plt.close(fig)


def array_to_rgba(grid, mask_grid, cmap_name='RdYlGn', vmin=None, vmax=None):
    valid = mask_grid & ~np.isnan(grid)
    if vmin is None:
        vmin = np.nanmin(grid[valid]) if valid.any() else 0.0
    if vmax is None:
        vmax = np.nanmax(grid[valid]) if valid.any() else 1.0
    if vmin == vmax:
        vmax = vmin + 1e-6
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    cmap = mpl.colormaps[cmap_name]
    rgba = (cmap(norm(grid)) * 255).astype(np.uint8)
    rgba[~valid] = [0, 0, 0, 0]
    return rgba, vmin, vmax


# Which indices get a full cartographic map per parcel (in addition to the quick grid).
# Add more names from heatmap_indices below if you want every index mapped this way --
# each one triggers its own basemap fetch, so this default subset keeps runtime reasonable.
CARTOGRAPHIC_INDICES = ['NDVI', 'EVI', 'SAVI', 'NDWI', 'NDRE', 'MSI', 'GCI', 'NDMI', 'BSI', 'thermal']
CARTOGRAPHIC_REVERSE = {'MSI', 'SIPI', 'ARI', 'BI', 'BSI', 'CRI1_Proxy'}


---
## Download the block once

All 63 parcels lie inside one 2.2 × 1.3 km block, so the imagery is fetched
**once** and every parcel is clipped from it locally.

| | per parcel | whole block |
|---|---|---|
| Earth Engine requests | ~945 | **15** |
| Typical runtime | 1–3 hours | **2–5 minutes** |
| Quota risk | high | negligible |

The block is roughly 30,000 pixels at 10 m — far below the 48 MB
`getDownloadURL` limit.

In [ ]:
# ════════════════════════════════════════════════════════════════
#  ONE DOWNLOAD FOR THE WHOLE BLOCK
# ════════════════════════════════════════════════════════════════
minx, miny, maxx, maxy = parcels_gdf.total_bounds
bd = meters_to_deg(BLOCK_BUFFER_M, parcels_gdf['lat'].mean())
block_aoi = ee.Geometry.Rectangle([minx - bd, miny - bd, maxx + bd, maxy + bd])

est_px = ((maxx - minx + 2*bd) * 111320 * np.cos(np.radians(parcels_gdf.lat.mean())) / SCALE_M) * \
         ((maxy - miny + 2*bd) * 110574 / SCALE_M)
print(f'Block AOI ~{est_px:,.0f} pixels at {SCALE_M} m  '
      f'({est_px*4/1e6:.1f} MB per band)\n')

# --- Sentinel-2 --------------------------------------------------
s2_img, s2_cloud = get_least_cloudy_image(
    'COPERNICUS/S2_SR_HARMONIZED', block_aoi, START_DATE, END_DATE,
    'CLOUDY_PIXEL_PERCENTAGE', CLOUD_THRESHOLDS_S2)
if s2_img is None:
    raise RuntimeError('No Sentinel-2 scene found. Widen the date window.')

BLOCK_S2_DATE = s2_img.date().format('YYYY-MM-dd').getInfo()
print(f'Sentinel-2 : {BLOCK_S2_DATE}  (cloud < {s2_cloud}%)')
block_s2 = download_bands(s2_img.clip(block_aoi), S2_BANDS, block_aoi,
                          SCALE_M, os.path.join(BLOCK_DIR, 'S2'))
if 'B4' not in block_s2:
    raise RuntimeError('B4 failed to download — cannot continue.')

# --- Landsat 8/9 land-surface temperature (Level 2, nearest to the S2 date) ----
block_thermal = None
ls_img, BLOCK_LS_DATE, ls_gap = get_landsat_lst_image(block_aoi, BLOCK_S2_DATE,
                                                       max_gap_days=LST_MAX_GAP_DAYS)
if ls_img is not None:
    print(f'Landsat LST: {BLOCK_LS_DATE}  gap {ls_gap} days'
          + ('   <-- stale, treat water stress with caution' if ls_gap > 4 else '   OK'))
    ls_paths = download_bands(ls_img.clip(block_aoi), ['LST_C'], block_aoi,
                              SCALE_M, os.path.join(BLOCK_DIR, 'LS8'))
    if 'LST_C' in ls_paths:
        block_thermal = resample_to_reference(
            ls_paths['LST_C'], block_s2['B4'],
            os.path.join(BLOCK_DIR, 'thermal_resampled.tif'), src_nodata=-999)
else:
    BLOCK_LS_DATE = None
    print(f'Landsat LST: no scene within {LST_MAX_GAP_DAYS} days - thermal will be NaN')

# --- Sentinel-1 ---------------------------------------------------
block_vv = block_vh = None
s1_img = get_sentinel1_image(block_aoi, START_DATE, END_DATE)
if s1_img is not None:
    print('Sentinel-1 : found')
    s1_paths = download_bands(s1_img.clip(block_aoi), ['VV','VH'], block_aoi,
                              SCALE_M, os.path.join(BLOCK_DIR, 'S1'))
    if 'VV' in s1_paths:
        block_vv = resample_to_reference(s1_paths['VV'], block_s2['B4'],
                        os.path.join(BLOCK_DIR, 'VV_resampled.tif'))
    if 'VH' in s1_paths:
        block_vh = resample_to_reference(s1_paths['VH'], block_s2['B4'],
                        os.path.join(BLOCK_DIR, 'VH_resampled.tif'))
else:
    print('Sentinel-1 : none found')

# --- assemble the shared band table -------------------------------
BLOCK_PATHS = dict(block_s2)
if block_thermal: BLOCK_PATHS['thermal'] = block_thermal
if block_vv:      BLOCK_PATHS['VV'] = block_vv
if block_vh:      BLOCK_PATHS['VH'] = block_vh

block_df, (BH, BW) = bands_to_dataframe(BLOCK_PATHS, ref_key='B4')
if 'thermal' not in block_df.columns:
    block_df['thermal'] = np.nan
block_df.loc[block_df['thermal'] < -100, 'thermal'] = np.nan      # -999 = no data
for c in ['VV','VH']:
    if c not in block_df.columns:
        block_df[c] = np.nan

# Indices and classification are computed ONCE for the whole block.
block_df = compute_indices(block_df)
BLOCK_SEASON = season_for_date(BLOCK_S2_DATE)
print(f'Season for thermal thresholds: {BLOCK_SEASON}  (image date {BLOCK_S2_DATE})')
block_df = classify_crop_issues(block_df, season=BLOCK_SEASON)

# Raster geometry: pixel size and the true raster EDGES (pixel centres are half a
# pixel inside), needed so map overlays line up with the basemap.
_lon_g = block_df['longitude'].values.reshape(BH, BW)
_lat_g = block_df['latitude'].values.reshape(BH, BW)
BLOCK_DX = float(_lon_g[0, 1] - _lon_g[0, 0])
BLOCK_DY = float(_lat_g[0, 0] - _lat_g[1, 0])
BLOCK_LON_E = (float(_lon_g.min()) - BLOCK_DX / 2, float(_lon_g.max()) + BLOCK_DX / 2)
BLOCK_LAT_E = (float(_lat_g.min()) - BLOCK_DY / 2, float(_lat_g.max()) + BLOCK_DY / 2)

# Point geometry for the spatial join in the loop below
block_pts = gpd.GeoSeries(
    gpd.points_from_xy(block_df['longitude'], block_df['latitude']),
    crs='EPSG:4326')

print(f'\nBlock raster : {BW} x {BH} = {len(block_df):,} pixels')
print('Indices and classification computed once for the whole block.')

In [ ]:
results = []
thumbnails = {}   # parcel_uid -> base64 PNG

for i, row in parcels_gdf.iterrows():
    uid = row['parcel_uid']
    print(f"\n=== {uid}  ({row['mahalle']} / {row['mevkii']}, {row['area_ha']} ha) "
          f"[{i+1}/{len(parcels_gdf)}] ===")

    parcel_dir = os.path.join(OUT_DIR, uid)
    os.makedirs(parcel_dir, exist_ok=True)
    s2_date = BLOCK_S2_DATE

    # ---- clip this parcel out of the block that is already in memory ----
    # A half-pixel buffer keeps boundary pixels that the polygon only clips.
    geom_buffered = row.geometry.buffer(meters_to_deg(SCALE_M, row['lat']) * 0.75)
    inside = block_pts.within(geom_buffered).values

    n_in = int(inside.sum())
    if n_in == 0:
        print('  no pixels fall inside this parcel — skipping')
        continue
    print(f'  {n_in} pixels inside the parcel')

    df = block_df                 # full block, for reshaping to the grid
    df_parcel = block_df.loc[inside].copy()
    mask_grid = inside.reshape(BH, BW)
    h, w = BH, BW
    # Tight window around THIS parcel (the maps used to cover the whole block, so a
    # 2.7 ha field was a speck in a ~9 km frame).
    _rr, _cc = np.where(mask_grid)
    r0, r1, c0, c1 = _rr.min(), _rr.max() + 1, _cc.min(), _cc.max() + 1
    win = (slice(r0, r1), slice(c0, c1))

    # --- Per-parcel pixel-level heatmaps: one PNG with every index, masked to the
    #     true parcel boundary (pixels outside it are left blank/NaN) ---
    heat_dir = os.path.join(parcel_dir, 'heatmaps')
    os.makedirs(heat_dir, exist_ok=True)
    heatmap_indices = ['NDVI', 'EVI', 'SAVI', 'OSAVI', 'MSAVI', 'NDWI', 'PRI_Proxy',
                        'CRI1_Proxy', 'NDRE', 'MSI', 'SIPI', 'GCI', 'MCARI', 'NDMI',
                        'GVMI', 'ARI', 'BI', 'BSI', 'thermal']
    reverse_cmap = {'CRI1_Proxy', 'MSI', 'SIPI', 'ARI', 'BI', 'BSI'}
    avail_idx = [c for c in heatmap_indices if c in df.columns]
    ncols_h = 5
    nrows_h = -(-len(avail_idx) // ncols_h)
    fig, axes = plt.subplots(nrows_h, ncols_h, figsize=(4 * ncols_h, 4 * nrows_h))
    axes = np.atleast_1d(axes).flatten()
    for ax, col in zip(axes, avail_idx):
        grid = df[col].values.reshape(h, w)[win].copy()
        grid[~mask_grid[win]] = np.nan
        cmap = 'RdYlGn_r' if col in reverse_cmap else 'RdYlGn'
        im = ax.imshow(grid, cmap=cmap)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
        ax.set_title(col, fontsize=10)
        ax.axis('off')
    for ax in axes[len(avail_idx):]:
        ax.set_visible(False)
    fig.suptitle(f"Parcel {row['ada']} — index heatmaps ({s2_date})")
    plt.tight_layout()
    heatmap_path = os.path.join(heat_dir, f'{uid}_index_heatmaps.png')
    plt.savefig(heatmap_path, dpi=130, bbox_inches='tight')
    plt.show()   # <-- display inline as the loop runs, not just save to disk
    plt.close(fig)

    # --- Per-parcel summary ---
    counts = df_parcel['prediction'].value_counts(normalize=True) * 100
    dominant = counts.idxmax()
    summary = {
        'parcel_uid': uid, 'ada': row['ada'], 'parsel': row['parsel'],
        'mahalle': row['mahalle'], 'mevkii': row['mevkii'], 'nitelik': row['nitelik'],
        'area_ha': row['area_ha'], 'lon': row['lon'], 'lat': row['lat'],
        'image_date': s2_date, 'n_pixels': len(df_parcel),
        'thermal_mean_C': df_parcel['thermal'].mean(),
        'dominant_class': dominant,
        'pct_healthy': counts.get('healthy', 0.0),
        'pct_bare_soil': counts.get('bare_soil', 0.0),
        'pct_water_stress': counts.get('water_stress', 0.0),
        'pct_open_water': counts.get('open_water', 0.0),
        'pct_nutrient_def': counts.get('nutrient_def or low_veg', 0.0),
        'pct_disease': counts.get('disease', 0.0),
        'pct_empty': counts.get('empty', 0.0),
    }
    # mean of every vegetation/soil/moisture index for this parcel
    for idx_col in ['NDVI', 'EVI', 'SAVI', 'OSAVI', 'MSAVI', 'NDWI', 'PRI_Proxy',
                     'CRI1_Proxy', 'NDRE', 'MSI', 'SIPI', 'GCI', 'MCARI', 'NDMI',
                     'GVMI', 'ARI', 'BI', 'BSI']:
        summary[f'{idx_col}_mean'] = df_parcel[idx_col].mean() if idx_col in df_parcel.columns else np.nan
    results.append(summary)
    print(f"  {len(df_parcel)} px | dominant: {dominant} | healthy {summary['pct_healthy']:.0f}% "
          f"| bare {summary['pct_bare_soil']:.0f}% | disease {summary['pct_disease']:.0f}%")

    # --- Classification raster on a real satellite basemap, with area name + date in the title ---
    # dtype=object: pandas >= 3 returns a string-dtype array that cannot be reshaped / masked in 2-D
    pred_full = np.asarray(df['prediction'].values, dtype=object).reshape((h, w))
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for cls, hexcol in CLASS_COLORS.items():
        hexcol = hexcol.lstrip('#')
        rgbcol = tuple(int(hexcol[j:j+2], 16) for j in (0, 2, 4))
        rgb[pred_full == cls] = rgbcol
    class_rgba = np.dstack([rgb, np.where(mask_grid, 220, 0).astype(np.uint8)])[win]

    lonmin = BLOCK_LON_E[0] + c0 * BLOCK_DX
    lonmax = BLOCK_LON_E[0] + c1 * BLOCK_DX
    latmax = BLOCK_LAT_E[1] - r0 * BLOCK_DY
    latmin = BLOCK_LAT_E[1] - r1 * BLOCK_DY
    area_name = f"{row['mahalle']} — {row['mevkii']} (Parcel {row['ada']})"

    patches_full = [mpatches.Patch(color=c, label=l.replace('_', ' ').title()) for l, c in CLASS_COLORS.items()]
    cartographic_categorical_map(
        class_rgba, lonmin, lonmax, latmin, latmax,
        title=f"{area_name}\nHealth classification — {s2_date}",
        legend_handles=patches_full, context_factor=0.6,
        out_path=os.path.join(heat_dir, f'{uid}_health_classification_map.png'))

    # Binary healthy (green) vs. unhealthy/everything-else (red), same cartographic style
    healthy_grid = np.where(pred_full == 'healthy', 1.0, 0.0)[win]
    healthy_rgba, _, _ = array_to_rgba(healthy_grid, mask_grid[win], cmap_name='RdYlGn', vmin=0, vmax=1)
    cartographic_continuous_map(
        healthy_rgba, lonmin, lonmax, latmin, latmax,
        title=f"{area_name}\nHealthy vs. unhealthy — {s2_date}",
        cmap_name='RdYlGn', vmin=0, vmax=1, cbar_label='0 = unhealthy/other, 1 = healthy',
        context_factor=0.6,
        out_path=os.path.join(heat_dir, f'{uid}_healthy_vs_unhealthy_map.png'))

    # One cartographic map per index — OFF by default.
    # 63 parcels x 19 indices would write ~1,200 PNG files.
    for col in (CARTOGRAPHIC_INDICES if MAKE_INDEX_MAPS else []):
        if col not in df.columns:
            continue
        grid = df[col].values.reshape(h, w)[win]
        if np.all(np.isnan(grid[mask_grid[win]])):
            continue
        cmap_name = 'RdYlGn_r' if col in CARTOGRAPHIC_REVERSE else 'RdYlGn'
        rgba, vmin, vmax = array_to_rgba(grid, mask_grid[win], cmap_name=cmap_name)
        cartographic_continuous_map(
            rgba, lonmin, lonmax, latmin, latmax,
            title=f"{area_name}\n{col} — {s2_date}",
            cmap_name=cmap_name, vmin=vmin, vmax=vmax, cbar_label=col, context_factor=0.6,
            out_path=os.path.join(heat_dir, f'{uid}_{col}_map.png'))

    thumb_fig, thumb_ax = plt.subplots(figsize=(2.5, 2.5))
    thumb_ax.imshow(rgb[win])
    thumb_ax.axis('off')
    buf = io.BytesIO()
    thumb_fig.savefig(buf, format='png', bbox_inches='tight', dpi=100)
    plt.close(thumb_fig)
    thumbnails[uid] = base64.b64encode(buf.getvalue()).decode('utf-8')

summary_df = pd.DataFrame(results)
summary_df


---
## All 63 parcels on one satellite map

The per-parcel cartographic maps zoom out to `context_factor = 1.5` with a
minimum pad of 0.0015°. On a 2.7 ha field that leaves the parcel as a speck in
the middle of a 4 km frame — which is what you were seeing.

These cells render the **whole block instead**: every parcel's pixel-level
classification in one figure, over the same Esri satellite basemap, with white
parcel outlines and numbers so each field stays identifiable.

Three versions:

| File | Shows |
|---|---|
| `block_health_classification.png` | Every pixel classified — the detailed view |
| `block_health_classification_detailed.png` | Same map, plus **owner, number of owners, registered area and mapped area** printed on every parcel under its number |
| `block_health_classification_detailed.pdf` | Vector version of the detailed map — text stays sharp at any zoom |
| `block_dominant_class.png` | Each parcel filled with its single dominant class — the briefing view |
| `block_<index>.png` | Any index across the whole block (NDVI by default) |

Run these **after** the main loop, so `block_df` and `summary_df` exist.

In [ ]:
# ════════════════════════════════════════════════════════════════
#  ALL PARCELS, ONE CARTOGRAPHIC MAP
#  High-resolution output + owner / owners / registered / mapped area
#  printed on every parcel under its number.
#
#  Needs (first cell of the notebook):  !pip -q install arabic-reshaper python-bidi
# ════════════════════════════════════════════════════════════════
import re, textwrap
from shapely.ops import unary_union
import matplotlib.patheffects as pe

# ---- output quality (edit these) -------------------------------------
SAVE_DPI     = 300   # PNG resolution of the detailed maps: 24x16 in -> 7200x4800 px
SAVE_PDF     = True  # also write a vector PDF: text stays sharp at any zoom level
BASEMAP_ZOOM = 18    # satellite tile zoom: 17 = faster, 19 = sharpest (slow, may be missing)
DETAIL_SIZE  = 6     # font size of the text printed on each parcel
plt.rcParams['pdf.fonttype'] = 42   # embed real fonts in the PDF

# ---- what goes on each parcel (set any to False to hide it) ----------
DETAIL_FIELDS = dict(owner=True, owner_count=True, area_registered=True, area_mapped=True)
OWNER_WRAP_CHARS = 14   # wrap long Arabic names to this many characters per line
OWNER_MAX_LINES  = 3    # names longer than this are cut with '...'

# ---- Arabic text support (matplotlib does not shape/reorder Arabic itself) ----
try:
    import arabic_reshaper
    from bidi.algorithm import get_display
    # DejaVu Sans (Colab's default font) lacks the single-glyph ligatures such as
    # the word 'Allah', so ligatures are switched off; letters still join correctly.
    _reshaper = arabic_reshaper.ArabicReshaper(configuration={'support_ligatures': False})

    def _ar(txt):
        return get_display(_reshaper.reshape(txt))
except ImportError:
    print('arabic-reshaper / python-bidi not installed -> Arabic names will look '
          'disconnected. Run:  !pip -q install arabic-reshaper python-bidi')

    def _ar(txt):
        return txt


def _has(v):
    return v is not None and not (isinstance(v, float) and np.isnan(v)) and str(v).strip() != ''


# ---- clean the owner text ---------------------------------------------
# For subdivided (multi-owner) parcels the KML popup text can carry extra junk in
# the owner field, e.g. '— 35 —', '|15|' or '0.014914 km2'. Keep only the Arabic
# words; separators ( | ; / — – , ، newline ) become ' / ' between owners.
_SEP        = re.compile(r'[|;/\n\r\u2014\u2013,\u060c]+')
_NON_ARABIC = re.compile(r'[^\u0621-\u064A\s]')


def clean_owner(raw):
    if not _has(raw):
        return ''
    parts = []
    for piece in _SEP.split(str(raw)):
        piece = re.sub(r'\s+', ' ', _NON_ARABIC.sub(' ', piece)).strip()
        if piece:
            parts.append(piece)
    return ' / '.join(parts)


if 'owner' in parcels_gdf.columns and 'owner_raw' not in parcels_gdf.columns:
    parcels_gdf['owner_raw'] = parcels_gdf['owner']                  # keep the original
    _clean = parcels_gdf['owner_raw'].map(clean_owner)
    parcels_gdf['owner'] = [c if c else o for c, o in zip(_clean, parcels_gdf['owner_raw'])]
    _chg = parcels_gdf[parcels_gdf['owner'] != parcels_gdf['owner_raw']]
    print(f'owner text cleaned on {len(_chg)} parcel(s) (original kept in column owner_raw):')
    for _, r in _chg.head(12).iterrows():
        print(f"   parcel {r['ada']:>3}:  {r['owner_raw']!r}\n              ->  {r['owner']!r}")


def _detail_text(r):
    """Multi-line label for one parcel: owner, no. of owners, registered and mapped area."""
    lines = []
    if DETAIL_FIELDS.get('owner'):
        name = str(r.get('owner')).strip() if _has(r.get('owner')) else '-'
        for ln in textwrap.wrap(name, width=OWNER_WRAP_CHARS,
                                max_lines=OWNER_MAX_LINES, placeholder='...') or ['-']:
            lines.append(_ar(ln))
    if DETAIL_FIELDS.get('owner_count') and _has(r.get('owner_count')):
        n = int(r['owner_count'])
        lines.append(f"{n} owner{'s' if n != 1 else ''}")
    if DETAIL_FIELDS.get('area_registered') and _has(r.get('area_feddan_excel')):
        lines.append(f"Reg {r['area_feddan_excel']:.2f} fd")
    if DETAIL_FIELDS.get('area_mapped') and _has(r.get('area_feddan_polygon')):
        lines.append(f"Map {r['area_feddan_polygon']:.2f} fd")
    return '\n'.join(lines)


def _draw_parcel_labels(ax, label_size=6, detail=False, detail_size=DETAIL_SIZE):
    """Parcel number at each centroid; with detail=True the owner data goes underneath."""
    halo = [pe.withStroke(linewidth=1.8, foreground='black')]
    for _, r in parcels_gdf.iterrows():
        c = r.geometry.centroid
        if not detail:
            ax.text(c.x, c.y, str(r['ada']), fontsize=label_size, ha='center',
                    va='center', color='white', zorder=5, path_effects=halo)
            continue
        ax.annotate(str(r['ada']), xy=(c.x, c.y), xytext=(0, 1),
                    textcoords='offset points', fontsize=label_size + 3,
                    fontweight='bold', ha='center', va='bottom', color='white',
                    zorder=6, path_effects=halo)
        txt = _detail_text(r)
        if txt:
            ax.annotate(txt, xy=(c.x, c.y), xytext=(0, -1),
                        textcoords='offset points', fontsize=detail_size,
                        ha='center', va='top', color='white', linespacing=1.15,
                        zorder=6,
                        bbox=dict(boxstyle='round,pad=0.18', fc='black',
                                  ec='none', alpha=0.62))


# ---- block geometry ----------------------------------------------------
# Union of every parcel, half-pixel buffer so boundary pixels survive
_buf = meters_to_deg(SCALE_M, parcels_gdf['lat'].mean()) * 0.75
BLOCK_UNION = unary_union([g.buffer(_buf) for g in parcels_gdf.geometry])
BLOCK_INSIDE = block_pts.within(BLOCK_UNION).values
BLOCK_MASK = BLOCK_INSIDE.reshape(BH, BW)

BLOCK_LON, BLOCK_LAT = BLOCK_LON_E, BLOCK_LAT_E      # true raster edges (half a pixel beyond the centres)
BLOCK_NAME = f"{parcels_gdf['mahalle'].iloc[0]} — {parcels_gdf['mevkii'].iloc[0]}"

print(f'pixels inside a parcel: {BLOCK_INSIDE.sum():,} of {len(block_df):,} '
      f'({BLOCK_INSIDE.sum()/len(block_df)*100:.0f}% of the block)')


def _add_basemap(ax):
    """Esri satellite tiles at BASEMAP_ZOOM; falls back to automatic zoom if that fails."""
    try:
        import contextily as cx
    except Exception as e:
        print(f'    basemap not added ({e})'); return
    for z in (BASEMAP_ZOOM, 'auto'):
        try:
            cx.add_basemap(ax, crs='EPSG:4326', source=cx.providers.Esri.WorldImagery,
                           zoom=z, attribution=False)
            return
        except Exception as e:
            print(f'    basemap zoom={z} failed ({e})')


def _block_axes(rgba, title, context_frac=0.06, figsize=(16, 11),
                outline=True, labels=True, label_size=6,
                detail=False, detail_size=DETAIL_SIZE):
    """Satellite basemap zoomed to the block, overlay on top."""
    lonmin, lonmax = BLOCK_LON
    latmin, latmax = BLOCK_LAT
    pad_lon = (lonmax - lonmin) * context_frac
    pad_lat = (latmax - latmin) * context_frac
    xlim = (lonmin - pad_lon, lonmax + pad_lon)
    ylim = (latmin - pad_lat, latmax + pad_lat)

    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    _add_basemap(ax)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)

    ax.imshow(rgba, extent=[lonmin, lonmax, latmin, latmax], origin='upper',
              interpolation='none', zorder=3)

    if outline:
        parcels_gdf.boundary.plot(ax=ax, edgecolor='white', linewidth=0.8, zorder=4)
    if labels:
        _draw_parcel_labels(ax, label_size, detail, detail_size)

    ax.set_title(title, fontsize=14, pad=14)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, p: _deg_to_dm(v, True)))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: _deg_to_dm(v, False)))
    plt.setp(ax.get_xticklabels(), fontsize=8)
    plt.setp(ax.get_yticklabels(), fontsize=8)
    ax.grid(True, color='white', linewidth=0.4, alpha=0.5, zorder=6)
    _add_north_arrow(ax)
    return fig, ax


def _save(fig, out_path, detail=False):
    """PNG at SAVE_DPI for detailed maps (200 dpi otherwise) + optional vector PDF."""
    fig.savefig(out_path, dpi=SAVE_DPI if detail else 200, bbox_inches='tight')
    print(f'saved -> {out_path}')
    if detail and SAVE_PDF:
        pdf_path = os.path.splitext(out_path)[0] + '.pdf'
        fig.savefig(pdf_path, bbox_inches='tight')
        print(f'saved -> {pdf_path}   (vector: zoom in as far as you like)')


def block_classification_map(out_path, labels=True, label_size=6,
                             detail=False, detail_size=DETAIL_SIZE):
    """Every pixel in every parcel, classified."""
    # np.asarray(..., dtype=object) is required: pandas >= 3.0 returns a
    # StringDtype array whose 2-D boolean mask indexing is broken.
    pred = np.asarray(block_df['prediction'].values, dtype=object).reshape(BH, BW)

    rgb = np.zeros((BH, BW, 3), dtype=np.uint8)
    for cls, hx in CLASS_COLORS.items():
        h = hx.lstrip('#')
        rgb[pred == cls] = tuple(int(h[j:j+2], 16) for j in (0, 2, 4))
    rgba = np.dstack([rgb, np.where(BLOCK_MASK, 235, 0).astype(np.uint8)])

    fig, ax = _block_axes(
        rgba, f'{BLOCK_NAME}\nHealth classification, all {len(parcels_gdf)} parcels'
              f' — {BLOCK_S2_DATE}', labels=labels, label_size=label_size,
        detail=detail, detail_size=detail_size,
        figsize=(24, 16) if detail else (16, 11))

    present = [c for c in CLASS_COLORS if c in set(pred[BLOCK_MASK])]
    ax.legend(handles=[mpatches.Patch(color=CLASS_COLORS[c],
                       label=c.replace('_', ' ').replace(' or ', ' / ').title())
                       for c in present],
              bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True, title='Legend')
    plt.tight_layout(); _save(fig, out_path, detail)
    plt.show(); plt.close(fig)


def block_dominant_map(out_path, labels=True, label_size=7,
                       detail=False, detail_size=DETAIL_SIZE):
    """Each parcel filled with its single dominant class — cleaner for a briefing."""
    dom = dict(zip(summary_df['parcel_uid'], summary_df['dominant_class']))
    fig, ax = _block_axes(np.zeros((1, 1, 4), dtype=np.uint8),
                          f'{BLOCK_NAME}\nDominant class per parcel — {BLOCK_S2_DATE}',
                          outline=False, labels=False,
                          figsize=(24, 16) if detail else (16, 11))
    for _, r in parcels_gdf.iterrows():
        col = CLASS_COLORS.get(dom.get(r['parcel_uid'], 'empty'), '#CCCCCC')
        gpd.GeoSeries([r.geometry], crs='EPSG:4326').plot(
            ax=ax, color=col, edgecolor='white', linewidth=0.9, alpha=0.85, zorder=3)
    if labels:
        _draw_parcel_labels(ax, label_size, detail, detail_size)

    present = [c for c in CLASS_COLORS if c in set(dom.values())]
    ax.legend(handles=[mpatches.Patch(color=CLASS_COLORS[c],
                       label=c.replace('_', ' ').replace(' or ', ' / ').title())
                       for c in present],
              bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True, title='Legend')
    plt.tight_layout(); _save(fig, out_path, detail)
    plt.show(); plt.close(fig)


def block_index_map(col, out_path, reverse=False, labels=False):
    """Any index across the whole block."""
    if col not in block_df.columns:
        print(f'    {col} not in block_df — skipped'); return
    grid = block_df[col].values.reshape(BH, BW)
    cmap = 'RdYlGn_r' if reverse else 'RdYlGn'
    rgba, vmin, vmax = array_to_rgba(grid, BLOCK_MASK, cmap_name=cmap)

    fig, ax = _block_axes(rgba, f'{BLOCK_NAME}\n{col} — {BLOCK_S2_DATE}',
                          labels=labels)
    sm = mpl.cm.ScalarMappable(cmap=cmap,
                               norm=mpl.colors.Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])
    fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.02, label=col)
    plt.tight_layout(); _save(fig, out_path, detail=False)
    plt.show(); plt.close(fig)


# ---- produce the maps -------------------------------------------------
block_classification_map(os.path.join(OUT_DIR, 'block_health_classification.png'))
block_classification_map(os.path.join(OUT_DIR, 'block_health_classification_detailed.png'),
                         detail=True)
block_dominant_map(os.path.join(OUT_DIR, 'block_dominant_class.png'))
block_dominant_map(os.path.join(OUT_DIR, 'block_dominant_class_detailed.png'), detail=True)

# Any index you want across the block. Add more names to this list.
for col in ['NDVI']:
    block_index_map(col, os.path.join(OUT_DIR, f'block_{col}.png'),
                    reverse=col in CARTOGRAPHIC_REVERSE)

# ---- download the high-resolution files (Colab) -----------------------
# from google.colab import files
# files.download(os.path.join(OUT_DIR, 'block_health_classification_detailed.png'))
# files.download(os.path.join(OUT_DIR, 'block_health_classification_detailed.pdf'))


In [ ]:
# Attach the KML attributes to the satellite results.
attrs = ['parcel_uid', 'ada', 'owner', 'owner_count',
         'area_ha_excel', 'area_feddan_excel',
         'area_ha_polygon', 'area_feddan_polygon',
         'geometry_vs_excel_pct', 'lat', 'lon']
attrs = [c for c in attrs if c in parcels_gdf.columns]

summary_df = summary_df.merge(parcels_gdf[attrs], on='parcel_uid',
                              how='left', suffixes=('', '_kml'))

lead = ['ada', 'parcel_uid', 'owner', 'area_feddan_excel',
        'dominant_class', 'pct_healthy']
summary_df = summary_df[[c for c in lead if c in summary_df.columns] +
                        [c for c in summary_df.columns if c not in lead]]

csv_path = 'nubaria_parcel_health_summary.csv'
summary_df.to_csv(csv_path, index=False, encoding='utf-8-sig')  # BOM: Arabic in Excel
print(f'saved -> {csv_path}   ({len(summary_df)} parcels, '
      f'{len(summary_df.columns)} columns)')
summary_df.head(10)

### Look up any parcel by the number on the map

The classification image labels each field with its parcel number.
`parcel_info(32)` returns everything known about that field — owner,
registered area, mapped area, and what the satellite found.

In [ ]:
def parcel_info(n, verbose=True):
    """
    Everything known about one parcel, keyed by the number printed on the map.

        parcel_info(32)

    Returns a pandas Series; also prints a readable block.
    """
    n = str(n).strip().lstrip('P').lstrip('0') or '0'

    row = parcels_gdf[parcels_gdf['ada'] == n]
    if row.empty:
        print(f'No parcel numbered {n}. Valid: 1 to {parcels_gdf.ada.astype(int).max()}')
        return None
    r = row.iloc[0]

    sat = {}
    if 'summary_df' in globals():
        s = summary_df[summary_df['parcel_uid'] == r['parcel_uid']]
        if not s.empty:
            sat = s.iloc[0].to_dict()

    if verbose:
        print('=' * 58)
        print(f'  PARCEL {n}      ({r["parcel_uid"]})')
        print('=' * 58)
        print(f'  Owner              : {r.get("owner", "—")}')
        if pd.notna(r.get('owner_count')):
            print(f'  Registered owners  : {int(r["owner_count"])}')
        print(f'  Registered area    : {r.get("area_feddan_excel", float("nan")):.2f} feddan'
              f'  ({r.get("area_ha_excel", float("nan")):.3f} ha)')
        print(f'  Mapped area        : {r["area_feddan_polygon"]:.2f} feddan'
              f'  ({r["area_ha_polygon"]:.3f} ha)')
        if pd.notna(r.get('geometry_vs_excel_pct')):
            d = r['geometry_vs_excel_pct']
            flag = '   <-- boundary may be wrong' if abs(d) > 10 else ''
            print(f'  Mapped vs registered: {d:+.2f}%{flag}')
        print(f'  Centre             : {r["lon"]:.5f} E, {r["lat"]:.5f} N')

        if sat:
            print('  ' + '-' * 54)
            print(f'  Dominant class     : {sat.get("dominant_class", "—")}')
            for k, lab in [('pct_healthy', '% healthy'),
                           ('pct_water_stress', '% water stress'),
                           ('pct_open_water', '% open water'),
                           ('pct_nutrient_def', '% nutrient def'),
                           ('pct_disease', '% disease'),
                           ('pct_bare_soil', '% bare soil')]:
                if k in sat and pd.notna(sat[k]):
                    print(f'  {lab:19s}: {sat[k]:.1f}')
            for k, lab in [('NDVI_mean', 'NDVI mean'),
                           ('NDRE_mean', 'NDRE mean'),
                           ('NDMI_mean', 'NDMI mean'),
                           ('thermal_mean_C', 'Canopy °C')]:
                if k in sat and pd.notna(sat[k]):
                    print(f'  {lab:19s}: {sat[k]:.3f}')
        else:
            print('\n  (run the processing loop to attach satellite results)')
        print('=' * 58)

    return pd.Series({**r.drop('geometry').to_dict(), **sat})


# Example — parcel 32 is the large red block in the centre of the map
_ = parcel_info(32)

### Owner report

One row per parcel, sorted worst-first — the order to visit them in.
Written with a UTF-8 BOM so Arabic names open correctly in Excel.

In [ ]:
rep = summary_df.copy()

cols = [('ada', 'Parcel'), ('owner', 'Owner'),
        ('owner_count', 'Owners'),
        ('area_feddan_excel', 'Area feddan (registered)'),
        ('area_feddan_polygon', 'Area feddan (mapped)'),
        ('geometry_vs_excel_pct', 'Area diff %'),
        ('dominant_class', 'Dominant class'),
        ('pct_healthy', '% healthy'), ('pct_water_stress', '% water stress'),
        ('pct_nutrient_def', '% nutrient def'), ('pct_disease', '% disease'),
        ('NDVI_mean', 'NDVI'), ('thermal_mean_C', 'Canopy C')]
cols = [(c, n) for c, n in cols if c in rep.columns]

report = rep[[c for c, _ in cols]].copy()
report.columns = [n for _, n in cols]
report = report.round(2)

out_csv = os.path.join(OUT_DIR, 'owner_report.csv') \
          if 'OUT_DIR' in globals() else 'owner_report.csv'
report.to_csv(out_csv, index=False, encoding='utf-8-sig')
print(f'saved -> {out_csv}')

if '% healthy' in report.columns:
    print('\n15 parcels to visit first:\n')
    display(report.sort_values('% healthy').head(15))

print('\n--- block totals ---')
print(f'parcels                 : {len(rep)}')
if 'area_feddan_excel' in rep.columns:
    print(f'registered area         : {rep.area_feddan_excel.sum():.1f} feddan')
print(f'mapped area             : {rep.area_feddan_polygon.sum():.1f} feddan')
if 'pct_healthy' in rep.columns:
    print(f'area-weighted % healthy : '
          f'{np.average(rep.pct_healthy, weights=rep.area_ha_polygon):.1f} %')

if 'dominant_class' in rep.columns:
    print()
    for cls in CLASS_COLORS:
        m = rep.dominant_class == cls
        if m.any():
            print(f'  {cls:26s} {int(m.sum()):3d} parcels   '
                  f'{rep.loc[m, "area_feddan_polygon"].sum():7.1f} feddan')

In [ ]:
merged = parcels_gdf.merge(summary_df, on='parcel_uid', suffixes=('', '_r'))

center_lat, center_lon = parcels_gdf['lat'].mean(), parcels_gdf['lon'].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=15, tiles='OpenStreetMap')
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri', name='Satellite', overlay=False).add_to(m)

for _, r in merged.iterrows():
    uid = r['parcel_uid']
    color = CLASS_COLORS.get(r['dominant_class'], '#808080')
    thumb_html = ''
    if uid in thumbnails:
        thumb_html = f"<img src='data:image/png;base64,{thumbnails[uid]}' width='140'><br>"
    popup_html = f'''
    <b>{r['mahalle']} — {r['mevkii']}</b><br>
    Parcel: {r['ada']}<br>
    Area: {r['area_ha']} ha | Image date: {r['image_date']}<br>
    {thumb_html}
    Dominant: <b style='color:{color}'>{r['dominant_class']}</b><br>
    Healthy {r['pct_healthy']:.0f}% | Bare soil {r['pct_bare_soil']:.0f}% |
    Water stress {r['pct_water_stress']:.0f}% | Nutrient/low-veg {r['pct_nutrient_def']:.0f}% |
    Disease {r['pct_disease']:.0f}%<br>
    NDVI {r['NDVI_mean']:.2f} | EVI {r['EVI_mean']:.2f} | SAVI {r['SAVI_mean']:.2f} |
    NDRE {r['NDRE_mean']:.2f} | NDWI {r['NDWI_mean']:.2f} | MSI {r['MSI_mean']:.2f} |
    Temp {r['thermal_mean_C']:.1f}°C
    '''
    folium.GeoJson(
        mapping(r.geometry),
        style_function=lambda x, color=color: {'fillColor': color, 'color': color,
                                                'weight': 2, 'fillOpacity': 0.6},
        tooltip=f"Parcel {r['ada']} — {r['dominant_class']}",
        popup=folium.Popup(popup_html, max_width=300)
    ).add_to(m)

legend_html = '''
<div style="position: fixed; bottom: 30px; left: 30px; z-index:9999; background:white;
            padding:10px; border:2px solid grey; border-radius:6px; font-size:13px;">
<b>Crop condition</b><br>
''' + ''.join(
    f"<span style='background:{c};display:inline-block;width:12px;height:12px;margin-right:6px;'></span>{label.replace('_',' ').title()}<br>"
    for label, c in CLASS_COLORS.items()
) + '</div>'
m.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl().add_to(m)
m.save('nubaria_crop_health_map.html')
m


In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))
merged['color'] = merged['dominant_class'].map(CLASS_COLORS)

# True-scale polygon fill (visible for the larger parcels)...
merged.plot(ax=ax, color=merged['color'], edgecolor='black', linewidth=0.6, zorder=2)

# ...plus a fixed-size marker at each centroid, since most parcels are only a few
# hectares; keeps them visible when zoomed out.
centroids = merged.geometry.centroid
ax.scatter(centroids.x, centroids.y, c=merged['color'], s=180,
           edgecolor='black', linewidth=0.8, zorder=3)

basemap_ok = False
try:
    import contextily as cx
    cx.add_basemap(ax, crs=merged.crs.to_string(), source=cx.providers.OpenStreetMap.Mapnik)
    basemap_ok = True
except Exception as e:
    print(f'Basemap not added ({e}). Install/enable internet access and re-run this cell.')

# Pad the view a bit so edge parcels and their labels aren't clipped
minx, miny, maxx, maxy = merged.total_bounds
# The block is only ~2 km across, so enforce a minimum pad
MIN_PAD_DEG = 0.002
pad_x = max((maxx - minx) * 0.15, MIN_PAD_DEG)
pad_y = max((maxy - miny) * 0.15, MIN_PAD_DEG)
ax.set_xlim(minx - pad_x, maxx + pad_x)
ax.set_ylim(miny - pad_y, maxy + pad_y)

# Labels with automatic de-overlap where parcels cluster together
texts = []
for _, r in merged.iterrows():
    c = r.geometry.centroid
    texts.append(ax.text(c.x, c.y, f"{r['ada']}", fontsize=7, ha='center', va='bottom'))
try:
    from adjustText import adjust_text
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))
except Exception:
    pass  # adjustText not available: labels may overlap slightly for close-together parcels

patches = [mpatches.Patch(color=c, label=l.replace('_', ' ').title()) for l, c in CLASS_COLORS.items()]
ax.legend(handles=patches, bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_title('Nubaria Parcels — Crop Health Overview (New Delta, Egypt)'
             + ('' if basemap_ok else '  [basemap unavailable]'))
ax.set_axis_off()
plt.tight_layout()
plt.savefig('nubaria_overview_map.png', dpi=200, bbox_inches='tight')
plt.show()


## Notes

### What changed

| Cell | Change |
|---|---|
| 3 | Earth Engine project from `EE_PROJECT_ID` (default `larvest-zeyad`); silent when credentials are cached |
| 4 | **KML loader** - owner names, registered areas, QA field |
| 5 | Egyptian season dates, `BLOCK_BUFFER_M`, `MAKE_INDEX_MAPS`, `LST_MAX_GAP_DAYS` |
| 6 | **NEW** `get_landsat_lst_image` - Landsat 8/9 *Level-2* land-surface temperature, scene nearest the Sentinel-2 date |
| 8 | Classifier corrections 1-7 (7 = canals get their own `open_water` class); `season_for_date` |
| 10-11 | Download the block once; thermal is now real LST (deg C), season follows the image date |
| 12 | Per-parcel maps/heatmaps are cropped to the parcel (they used to cover the whole block); pandas >= 3 safe |
| 20-21 | Web map / overview map retitled for Nubaria (they still said "Odunpazari, Turkiye") |
| all | Saved outputs removed (they were 38 MB of images from the old Turkish project) |

---

### Why the block download matters

Per-parcel downloading needs **63 x 15 = 945** Earth Engine requests.
Block downloading needs **15**. Same result, roughly 60x less work, and no
realistic quota risk.

This only works because the parcels are contiguous. If you later add parcels
tens of kilometres away, either widen the block (watch the 48 MB cap) or group
them into blocks and loop over the groups.

---

### Two things to check on every run

**Scene date.** Cell 11 picks the least-cloudy Sentinel-2 scene in the window, which
can be any date in it. The printed date decides the crop stage the classifier sees:
a pre-planting or post-harvest date will read as bare soil or disease. Narrow
`START_DATE` / `END_DATE` around the growth stage you care about.

**Thermal date gap.** Cell 11 prints the gap between the Sentinel-2 and Landsat
dates. Beyond about 4 days the thermal layer describes different weather from the
optical bands and the water-stress class becomes unreliable. Landsat thermal is
100 m native, so a 2.7 ha parcel contains only 1-4 independent thermal pixels:
treat it as a per-field signal, not per-pixel.

---

### Block-wide maps (all parcels in one figure)

| File | Shows |
|---|---|
| `block_health_classification.png` | Every pixel in every parcel, classified, on satellite |
| `block_dominant_class.png` | One colour per parcel — cleaner for a briefing slide |
| `block_NDVI.png` | Any index across the whole block |

**Why the per-parcel maps used to look so zoomed out:** `_setup_cartographic_axes`
pads by `context_factor = 1.5` with a 0.0015° floor. On a 2.7 ha parcel
(~0.002° wide) that pad is larger than the field itself, so the parcel
becomes a speck. The block maps use `context_frac = 0.06` against the whole
2.2 km block instead.

If you still want readable single-parcel maps, pass a smaller
`context_factor` (0.3 or so) to `_setup_cartographic_axes`.

---

### KML parcel source

Cell 4 reads `Nubarya_63_Parcels_GroundTruth.kml` instead of the embedded CSV,
bringing in owner names, registered areas and the QA field.

Owner names are **not** in `ExtendedData` — they exist only inside the HTML
popup, so `_owner_from_description()` extracts them by regex on the `Owner(s)`
table row. If the KML is regenerated with a different popup layout, that regex
is the thing to re-check.

**Three parcels where the drawn boundary disagrees with the registered area:**

| Parcel on map | Registered | Mapped | Difference |
|---|---|---|---|
| 16 | 22,025 m² | 27,189 m² | **+23.4 %** |
| 35 | 23,601 m² | 26,862 m² | **+13.8 %** |
| 62 | 16,832 m² | 32,569 m² | **+93.5 %** |

Parcel 62 is the one to resolve first — the polygon is nearly double the
registered area, so it is probably capturing a neighbouring field. Anything
currently reported for parcel 62 describes more land than the owner holds.

**Eight parcels have more than one registered owner** (5, 15, 16, 34, 35, 37,
44, 55). The classification covers the whole polygon, not each owner's share.
Per-owner figures would need the subdivisions drawn as separate geometry — the
KML records the split in the owner text but not in the shapes.

**One owner holds two parcels** (17 and 22).

All CSV exports use `utf-8-sig` so Arabic opens correctly in Excel.

---

### Classifier corrections (cell 8)

**What changed in the scoring**

```python
# nutrient — SIPI direction corrected
ramp(SIPI, 1.05, 1.70)            # was (1.8 - SIPI)

# nutrient — damped where tissue is dying
nutrient *= (1 - 0.55 * ramp(PSRI, 0.18, 0.38))

# bare soil — real BSI thresholds per soil class
ramp(BSI, SOIL['BSI']-0.10, SOIL['BSI']+0.15)     # was (BSI-100)/50

# disease — PRI removed, PSRI weighted twice, rescaled to reach 1.0
inv(NDVI, 0.75, 0.30), ramp(PSRI,0.18,0.38) x2, ramp(ARI,2,7), inv(EVI,0.55,0.20)

# water stress — NDMI counted twice, season-aware thermal
ramp(thermal, 34, 42) + inv(NDMI,0.20,-0.10) x2 + ...

# open water — now applied LAST so the bare override cannot cancel it
```

**Why NDRE is excluded from the disease score.** Low NDRE is the *nutrient*
signature. Including it in both classes made them indistinguishable, which is
part of why disease never fired. Senescence (PSRI) is what actually separates
dying tissue from hungry tissue, so PSRI carries the weight instead.

**Why `nutrient` is damped by PSRI.** A nitrogen-starved plant is pale but
alive, so PSRI stays low. Rising PSRI means tissue is genuinely dying. Without
this damping the two classes stay nearly identical and nutrient always wins.

**Idempotent scaling.** `compute_indices` now checks whether a band is still on
the DN scale before dividing. Re-running the cell no longer divides twice.

---

### Before publishing to the website

The confidence values (`*_prob`) are **relative weights, not calibrated
probabilities**. A pixel at `disease_prob = 0.6` does not mean 60 % of such
pixels are diseased. Do not display them as percentages to a farmer until they
are fitted against GPS-tagged scouting points.

If the map goes on larvest.ai, export **parcel id, class, percentages, indices,
date** only. Owner names are personal data — Larvest is Netherlands-registered,
so GDPR applies. They belong in `owner_report.csv` handed to the responsible
body, not on a public page.

## Re-viewing saved heatmaps later

If you close the notebook or scroll past the loop's output, use this cell to redisplay every
saved heatmap without re-running the Earth Engine downloads.


In [ ]:
from IPython.display import Image, display

for uid_dir in sorted(glob.glob(os.path.join(OUT_DIR, '*', 'heatmaps'))):
    for png in sorted(glob.glob(os.path.join(uid_dir, '*.png'))):
        print(png)
        display(Image(filename=png))
